# Presto first run on real project data

**Scope.** This notebook shows that the pinned, pretrained [Presto](https://github.com/nasaharvest/presto) model loads
and that its encoder runs on inputs built from this project's own data: ten Malawi 2023 field polygons, their
Sentinel-2 Collection 1 chips, and one monthly reflectance series per field.

**It proves that the model runs. It does not show that Presto predicts yield.** Nothing is trained or evaluated here;
yields appear only as labels in tables.

Before running (see README, *Presto first run*): `python -m pip install -e ".[dev]"`, `python scripts/get_presto.py`,
and the ten pilot chips from `scripts/download_sentinel2.py`. Restart the kernel and run all cells.

In [ ]:
from datetime import date

import numpy as np
import pandas as pd
import torch

from cropyield import ROOT
from cropyield.data.cohorts import load_cohort
from cropyield.features.presto_inputs import NORMED_BANDS, presto_inputs
from cropyield.models import presto_weights as weights
from cropyield.models.presto import EMBEDDING_SIZE, encode, load_presto, to_tensors
from cropyield.satellite.chips import read_chip
from cropyield.satellite.monthly import monthly_series

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
print("repository:", ROOT)
print("torch", torch.__version__, "| numpy", np.__version__, "| pandas", pd.__version__)

## 1. The ten records

`load_cohort` reads the screened audit CSV for the Malawi 2023 cohort in file order and gives every row a stable
`obs_id` (`dataset:source_id`). The pilot is its first ten rows, the same ten listed in
`configs/sentinel2_windows_malawi_2023_pilot.csv`. `read_chip` reads each record folder the downloader wrote.

In [ ]:
cohort = load_cohort("malawi_2023")
pilot = cohort.head(10).set_index("obs_id")

chips = {}
for run in sorted((ROOT / "data" / "imagery" / "sentinel2").glob("run-*")):
    for record_dir in sorted((run / "records").glob("malawi_2023__*")):
        if (record_dir / "record.json").exists():
            chip = read_chip(record_dir)
            if chip.obs_id in pilot.index:
                chips[chip.obs_id] = chip
missing = [obs_id for obs_id in pilot.index if obs_id not in chips]
assert not missing, f"run the downloader first; no chip folder for {missing}"

records = pd.DataFrame({
    "source_row": pilot["source_row"],
    "yield_kg_ha": pilot["yield_kg_ha"],
    "latitude": pilot["latitude"].round(4),
    "longitude": pilot["longitude"].round(4),
    "field_area_ha": pilot["field_area_ha"].astype(float).round(2),
    "season": pilot["season_start"] + " .. " + pilot["season_end"],
    "aoi": [chips[i].aoi_type for i in pilot.index],
    "aoi_pixels": [int(chips[i].aoi.sum()) for i in pilot.index],
    "status": [chips[i].status for i in pilot.index],
    "scenes": [len(chips[i].scenes) for i in pilot.index],
})
print(f"{len(cohort)} rows in the cohort, {len(pilot)} in the pilot, {len(chips)} chips read")
records

## 2. One record in detail: location, season window and accepted scenes

Each accepted scene passed the downloader's check of at least 80 % clear, valid pixels over the field. Reflectance in
the chips is unitless bottom-of-atmosphere reflectance with the Collection 1 offset already applied
(`docs/DECISIONS.md`, 26 October 2026).

In [ ]:
first_id = pilot.index[0]
first = chips[first_id]
row = pilot.loc[first_id]
season = (date.fromisoformat(row["season_start"]), date.fromisoformat(row["season_end"]))
print("obs_id        ", first.obs_id)
print("source row    ", first.source_row, "| yield", first.yield_kg_ha, "kg/ha")
print("location      ", f"lat {first.latitude:.5f}, lon {first.longitude:.5f}", "| grid", first.crs, first.aoi.shape)
print("season window ", season[0], "..", season[1], "| downloader window", first.start_date, "..", first.end_date, f"({first.window_basis})")
print("AOI           ", first.aoi_type, "|", int(first.aoi.sum()), "pixels of 10 m | entirely inside chip:", first.entire_aoi_inside_chip)
pd.DataFrame({
    "item_id": [s.item_id for s in first.scenes],
    "date": [s.date for s in first.scenes],
    "tile_cloud_%": [round(s.tile_cloud_cover, 1) for s in first.scenes],
    "clear_fraction_aoi": [round(s.clear_fraction_aoi, 3) for s in first.scenes],
    "valid_aoi_pixels": [int((first.aoi & s.valid).sum()) for s in first.scenes],
    "reflectance_min": [float(np.nanmin(s.reflectance)) for s in first.scenes],
    "reflectance_max": [float(np.nanmax(s.reflectance)) for s in first.scenes],
})

## 3. Monthly band values and missing months

`monthly_series` gives one row per calendar month of the season window: the median reflectance over the field pixels
that are valid in that month's scene. A month without an accepted scene is NaN and is listed as missing. The same
series, through its named bands, feeds the NDVI/GCVI ridge baseline later.

In [ ]:
series = monthly_series(first, *season)
monthly = pd.DataFrame(series.values, index=series.months, columns=series.bands)
monthly["NDVI"] = (series.band("B08") - series.band("B04")) / (series.band("B08") + series.band("B04"))
monthly["GCVI"] = series.band("B08") / series.band("B03") - 1
monthly["valid_pixels"] = series.n_pixels
monthly["scene"] = [i or "" for i in series.item_ids]
print("missing months:", [m for m, miss in zip(series.months, series.missing) if miss])
monthly.round(3)

## 4. Presto inputs from the series

`presto_inputs` places the ten Sentinel-2 bands into Presto's 17-band layout, computes NDVI the way Presto does, and
masks everything we do not have: Sentinel-1, ERA5 and SRTM at every timestep, and every band of a missing month.
Presto normalises Sentinel-2 digital numbers by dividing by 10 000; our reflectance is already on that scale, so the
value Presto sees equals the reflectance in the table above (checked below).

In [ ]:
inputs = presto_inputs(series, row["latitude"], row["longitude"])
print("x            ", inputs.x.shape, inputs.x.dtype, "(batch, months, 17 bands)")
print("mask         ", inputs.mask.shape, "| masked entries:", int(inputs.mask.sum()), "of", inputs.mask.size)
print("dynamic_world", inputs.dynamic_world.shape, "| unique values:", np.unique(inputs.dynamic_world).tolist(), "(9 = unknown, ignored)")
print("latlons      ", inputs.latlons.tolist(), "| first month index (0 = January):", inputs.month)
scale_check = np.allclose(inputs.x[0, ~series.missing, NORMED_BANDS.index("B8")], series.band("B08")[~series.missing])
print("x[B8] equals reflectance B08 for present months:", scale_check)
per_month = pd.DataFrame({
    "unmasked_bands": (inputs.mask[0] == 0).sum(axis=1).astype(int),
    "x_B4_red": inputs.x[0, :, NORMED_BANDS.index("B4")].round(3),
    "x_B8_nir": inputs.x[0, :, NORMED_BANDS.index("B8")].round(3),
    "x_NDVI": inputs.x[0, :, NORMED_BANDS.index("NDVI")].round(3),
    "masked_bands": [", ".join(b for b, m in zip(NORMED_BANDS, mask_row) if m == 1) for mask_row in inputs.mask[0]],
}, index=inputs.months)
per_month

## 5. Load the pinned checkpoint

The weights are `data/default_model.pt` from nasaharvest/presto at the pinned commit, downloaded by
`scripts/get_presto.py`; `load_presto` refuses a file whose size or SHA-256 differs and loads the state dict strictly
into the vendored model code (`src/cropyield/models/presto_single_file.py`, same commit).

In [ ]:
print("commit ", weights.PRESTO_COMMIT)
print("url    ", weights.PRESTO_WEIGHTS_URL)
print("path   ", weights.PRESTO_WEIGHTS_PATH.relative_to(ROOT), "|", weights.PRESTO_WEIGHTS_PATH.stat().st_size, "bytes")
print("sha256 ", weights.sha256(weights.PRESTO_WEIGHTS_PATH), "| matches pin:", weights.weights_verified())
model = load_presto()
n_encoder = sum(p.numel() for p in model.encoder.parameters())
n_total = sum(p.numel() for p in model.parameters())
print(f"loaded strictly | encoder parameters {n_encoder:,} | encoder + decoder {n_total:,} | eval mode: {not model.training}")

## 6. Run the encoder on all ten records

One record at a time: Presto's encoder requires every item of a batch to have the same number of masked tokens, and
the number of missing months differs between records. `encode` returns the mean over the kept tokens after the
transformer blocks, a 128-dimensional embedding per field.

In [ ]:
rows, embeddings = [], []
for obs_id in pilot.index:
    chip, r = chips[obs_id], pilot.loc[obs_id]
    s = monthly_series(chip, date.fromisoformat(r["season_start"]), date.fromisoformat(r["season_end"]))
    inp = presto_inputs(s, r["latitude"], r["longitude"])
    emb = encode(model, inp)
    embeddings.append(emb)
    rows.append({"months_with_data": int((~s.missing).sum()), "months_missing": int(s.missing.sum()),
                 "x_shape": tuple(inp.x.shape), "embedding_shape": emb.shape,
                 "finite": bool(np.isfinite(emb).all()), "norm": round(float(np.linalg.norm(emb)), 3),
                 "yield_kg_ha": r["yield_kg_ha"]})
embeddings = np.stack(embeddings)
summary = pd.DataFrame(rows, index=pilot.index)
print("embeddings", embeddings.shape, "| all finite:", bool(np.isfinite(embeddings).all()),
      "| all records produced an embedding:", len(embeddings) == len(pilot))
summary

In [ ]:
print("first 8 dimensions of each embedding:")
print(pd.DataFrame(embeddings[:, :8], index=[i.split(":")[1][:8] for i in pilot.index]).round(3))
unit = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
print("\ncosine similarity between the ten field embeddings (1 on the diagonal):")
print(pd.DataFrame(unit @ unit.T, index=range(1, 11), columns=range(1, 11)).round(2))

## 7. The official regression head attaches (untrained)

`construct_finetuning_model(num_outputs=1, regression=True)` is the head the research plan names. Its output here comes
from random initial weights and means nothing; the cell only shows that the shapes line up for later fine-tuning.

In [ ]:
head_model = model.construct_finetuning_model(num_outputs=1, regression=True).eval()
with torch.no_grad():
    prediction = head_model(**to_tensors(inputs))
print("untrained head output", tuple(prediction.shape), "| finite:", bool(torch.isfinite(prediction).all()))
print("trainable parameters in the fine-tuning model:", f"{sum(p.numel() for p in head_model.parameters() if p.requires_grad):,}")

## What this shows, and what it does not

Shown: the pinned checkpoint loads strictly into the pinned model code; real chips become a monthly series with the
expected missing months; Presto's inputs have the documented shapes and scale; the frozen encoder produces a finite
128-dimensional embedding for every pilot field; the regression head attaches.

Not shown: that these embeddings carry yield information. That needs the frozen-embedding baseline under the frozen
spatial splits, which is week-1 work in `docs/ROADMAP.md`.